In [ ]:
!pip install beautifulsoup4 pandas openpyxl requests pdfplumber

import requests
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import files
import os
import re
import difflib
import html
import pdfplumber

# --- KONFIGURATION ---
VERANTWORTUNGSBEREICHE = ["351", "21", "70", "Abteilung X", "Abteilung Y"]

def clean_sheet_name(name):
    if not name: return "MaRisk_Extraktion"
    return re.sub(r'[\\/*?:\[\]]', '_', name)[:31]

def clean_extracted_text(text):
    if pd.isna(text) or text == "[Nicht vorhanden]": return ""
    text = str(text)
    text = re.sub(r'([A-Za-zäöüß])-\s+(?!(?:und|oder|sowie|als|bzw|bis)\b)([A-Za-zäöüß])', r'\1\2', text)
    text = text.replace('- controlling', '-controlling')
    text = text.replace('Risiko controlling', 'Risikocontrolling')
    text = text.replace('Compliance -Funktion', 'Compliance-Funktion')
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'(?<!\n)\s+([a-z]\)\s|•\s)', r'\n\1', text)
    return text.strip()

def normalize_for_comparison(text):
    if pd.isna(text) or text == "[Nicht vorhanden]": return ""
    text = str(text).replace('\xad', '').replace('\u00ad', '')
    text = re.sub(r'[\s\n\r\t•·\*■-]', '', text)
    return text.lower()

def normalize_token(t):
    t = t.replace('\xad', '').replace('\u00ad', '')
    t = re.sub(r'[•·\*■]', '-', t)
    return t

def generate_match_key(abschnitt, tz):
    abs_str = " ".join(str(abschnitt).split())
    match = re.match(r'^([A-Z]{2,3}(?:\s+[\d\.]+)?)', abs_str)
    sec_key = match.group(1).strip() if match else abs_str
    tz_key = str(tz).replace('.0', '').strip()
    return (sec_key, tz_key)

def generate_inline_diff(old_text, new_text):
    old_tokens = [t for t in re.split(r'(\s+)', str(old_text)) if t]
    new_tokens = [t for t in re.split(r'(\s+)', str(new_text)) if t]
    matcher = difflib.SequenceMatcher(None, old_tokens, new_tokens, autojunk=False)
    result = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal': result.append("".join(new_tokens[j1:j2]))
        elif tag == 'delete': result.append(f'<del>{"".join(old_tokens[i1:i2])}</del>')
        elif tag == 'insert': result.append(f'<ins>{"".join(new_tokens[j1:j2])}</ins>')
        elif tag == 'replace': result.append(f'<del>{"".join(old_tokens[i1:i2])}</del><ins>{"".join(new_tokens[j1:j2])}</ins>')
    return "".join(result)

def format_text_with_lists(text):
    lines = text.split('\n')
    if len(lines) > 1:
        html_parts = [lines[0]]
        for line in lines[1:]:
            html_parts.append(f"<div style='margin-left: 20px; margin-top: 4px;'>{line.strip()}</div>")
        return "".join(html_parts)
    return text

def format_erlaeuterung_with_html(text):
    lines = text.split('\n')
    html_parts = []
    for line in lines:
        line = line.strip()
        if not line:
            continue

        is_bold = False
        if line.startswith('<b>') and line.endswith('</b>'):
            is_bold = True
            line = line[3:-4]

        line_escaped = html.escape(line)

        if line_escaped.startswith('•'):
            html_parts.append(f"<div style='margin-left: 15px;'>{line_escaped}</div>")
        elif is_bold:
            html_parts.append(f"<div style='margin-top: 14px; margin-bottom: 6px; font-size: 13px; font-weight: 700; color: #2d3748;'>{line_escaped}</div>")
        else:
            html_parts.append(f"<div style='margin-bottom: 6px;'>{line_escaped}</div>")

    return "".join(html_parts)

def build_lines_from_words(words):
    if not words: return []
    words.sort(key=lambda w: w['top'])
    lines = []
    current_line_words = [words[0]]
    current_top = words[0]['top']
    for w in words[1:]:
        if abs(w['top'] - current_top) <= 4:
            current_line_words.append(w)
        else:
            current_line_words.sort(key=lambda x: x['x0'])
            lines.append({"text": " ".join([x['text'] for x in current_line_words]), "top": current_top})
            current_line_words = [w]
            current_top = w['top']
    if current_line_words:
        current_line_words.sort(key=lambda x: x['x0'])
        lines.append({"text": " ".join([x['text'] for x in current_line_words]), "top": current_top})
    return lines

def is_junk_line(line):
    line_clean = line.strip()
    exact_matches = [
        "BaFin", "(BaFin)", "Erläuterungen", "mit Erläuterungen (BaFin)", "mit Erläuterungen",
        "Bundesanstalt für", "Finanzdienstleistungsaufsicht",
        "Bundesanstalt für Finanzdienstleistungsaufsicht",
        "Bundesanstalt für Finanzdienstleistungsaufsicht (BaFin)",
        "Bundesanstalt für Finanzdienstleis", "tungsaufsicht",
        "02/2026", "02/2024"
    ]
    if line_clean in exact_matches: return True
    if line_clean.startswith("Bundesanstalt für") and len(line_clean) < 55: return True
    if line_clean.startswith("Finanzdienstleistungsaufsicht") and len(line_clean) < 40: return True
    if line_clean.startswith("BaFin") and len(line_clean) < 10: return True

    if re.search(r'Seite\s+\d+\s+von\s+\d+', line_clean, re.IGNORECASE): return True
    if re.match(r'^\d{2}\.\d{2}\.\d{4}$', line_clean): return True
    if re.search(r'\.{4,}', line_clean): return True
    if re.match(r'^[\d\s]+$', line_clean): return True
    if "Entwurf der MaRisk" in line_clean and len(line_clean) < 150: return True
    if "Änderungsversion" in line_clean and len(line_clean) < 150: return True
    if "Fassung vom" in line_clean and len(line_clean) < 150: return True
    if "02/2024" in line_clean and "Fassung" in line_clean and len(line_clean) < 150: return True

    if re.search(r'BA\s*54\s*-\s*MaRisk', line_clean, re.IGNORECASE): return True
    if re.search(r'Konsultation(?:\s*\d{2}/\d{4})?', line_clean, re.IGNORECASE) and len(line_clean) < 40: return True
    if re.match(r'^\d{2}/\d{4}$', line_clean): return True
    if "xx.xx.2026" in line_clean: return True

    return False

def format_bullet(text):
    if re.match(r'^[-•·\*■]\s+', text):
        return '• ' + re.sub(r'^[-•·\*■]\s+', '', text)
    return text

def normalize_header(h):
    return re.sub(r'\W+', '', str(h)).lower()

def extract_from_pdf(pdf_path):
    data_list = []
    current_abschnitt = "Unbekannt"
    current_active_tz = None
    pending_erl_lines = []

    abschnitt_pattern = re.compile(r'^((?:AT|BTO|BTR|BT)(?:\s+\d+(?:\.\d+)*)?\s+[A-ZÄÖÜ].*)')
    tz_pattern = re.compile(r'^(\d{1,3})(?:\.|\s+)(.*)')

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            strike_lines = []
            for obj_type in ['lines', 'rects', 'curves']:
                for obj in getattr(page, obj_type, []):
                    h = obj.get('height', 0)
                    w = obj.get('width', 0)
                    if h < 4 and w > 5:
                        strike_lines.append(obj)

            def valid_char(obj):
                if obj.get('object_type') == 'char':
                    mid_y = (obj['top'] + obj['bottom']) / 2
                    mid_x = (obj['x0'] + obj['x1']) / 2
                    for s in strike_lines:
                        if s['top'] - 3 <= mid_y <= s['bottom'] + 3:
                            if s['x0'] <= mid_x <= s['x1']:
                                return False
                return True

            clean_page = page.filter(valid_char)
            words = clean_page.extract_words(x_tolerance=2, y_tolerance=3)
            page_midpoint = page.width * 0.52

            left_words = [w for w in words if w['x0'] < page_midpoint]
            right_words = [w for w in words if w['x0'] >= page_midpoint]

            left_lines = build_lines_from_words(left_words)
            right_lines = build_lines_from_words(right_words)

            carried_over_tz = current_active_tz
            page_sections = []

            for l_line in left_lines:
                text = l_line['text'].strip()
                top_y = l_line['top']

                if not text or is_junk_line(text): continue

                abs_match = abschnitt_pattern.match(text)
                if abs_match and len(text) < 150 and not text.endswith('.'):
                    if re.search(r'\s+\d+$', text): continue

                    new_abschnitt = abs_match.group(1).strip()

                    if normalize_header(new_abschnitt) == normalize_header(current_abschnitt):
                        continue

                    current_abschnitt = new_abschnitt
                    current_active_tz = None
                    pending_erl_lines.clear()
                    page_sections.append((top_y, "HEADER"))
                    continue

                tz_match = tz_pattern.match(text)
                is_new_tz = False

                if tz_match:
                    num_str = tz_match.group(1)
                    text_part = tz_match.group(2).strip()

                    if text_part.startswith('KWG') or text_part.startswith('Abs.') or text_part.startswith('Satz') or text_part.startswith('Mio') or re.match(r'^(Januar|Februar|März|April|Mai|Juni|Juli|August|September|Oktober|November|Dezember)', text_part, re.IGNORECASE):
                        is_new_tz = False
                    else:
                        if not current_active_tz and num_str in ["1", "2"]:
                            is_new_tz = True
                        elif current_active_tz and current_active_tz["TZ"].isdigit() and num_str.isdigit():
                            diff = int(num_str) - int(current_active_tz["TZ"])
                            if 1 <= diff <= 2:
                                is_new_tz = True

                if is_new_tz:
                    new_tz_item = {
                        "Abschnitt": current_abschnitt,
                        "TZ": tz_match.group(1),
                        "left": [format_bullet(tz_match.group(2).strip())],
                        "right": []
                    }
                    if pending_erl_lines:
                        new_tz_item["right"].extend(pending_erl_lines)
                        pending_erl_lines.clear()

                    data_list.append(new_tz_item)
                    page_sections.append((top_y, new_tz_item))
                    current_active_tz = new_tz_item
                else:
                    if current_active_tz:
                        current_active_tz["left"].append(format_bullet(text))

            for line_obj in right_lines:
                text = line_obj['text'].strip()
                top_y = line_obj['top']

                if not text or is_junk_line(text): continue

                target = None
                for sec_y, tz_item in reversed(page_sections):
                    if top_y >= sec_y - 15:
                        target = tz_item
                        break

                if target is None:
                    if carried_over_tz is not None:
                        target = carried_over_tz
                    else:
                        target = "PENDING"

                if target == "HEADER" or target == "PENDING":
                    pending_erl_lines.append(format_bullet(text))
                else:
                    target["right"].append(format_bullet(text))

    final_data = []
    for obj in data_list:
        right_text = ""
        prev_ended_sentence = True

        for r_line in obj["right"]:
            r_line = r_line.strip()
            if not r_line: continue

            if r_line.startswith('•'):
                right_text += f"\n{r_line}\n"
                prev_ended_sentence = True
            elif prev_ended_sentence and len(r_line) < 75 and r_line[0].isupper() and not r_line[-1] in ".,;:!?-":
                right_text += f"\n<b>{r_line}</b>\n"
                prev_ended_sentence = True
            else:
                if right_text and not right_text.endswith('\n') and not right_text.endswith(' '):
                    right_text += " "
                right_text += r_line

                if r_line[-1] in ".:!?":
                    prev_ended_sentence = True
                else:
                    prev_ended_sentence = False

        final_data.append({
            "Abschnitt": obj["Abschnitt"],
            "TZ": obj["TZ"],
            "Inhalt MaRisk": clean_extracted_text(" ".join(obj["left"])),
            "Erläuterung": clean_extracted_text(right_text)
        })

    return final_data

# --- 1. DATENERHEBUNG ---
print("="*40)
print("MaRisk-Novelle: Gap-Analyse V12.14 (Excel VBA Edition)")
print("="*40)

upload_choice = input("Bestands-Masterdatei (alte Fassung) hochladen? (j/n): ").strip().lower()
master_filename, old_sheet_name = None, ""
if upload_choice == 'j':
    print("\nBitte laden Sie die Excel-Masterdatei hoch:")
    uploaded = files.upload()
    if uploaded:
        master_filename = list(uploaded.keys())[0]
        old_sheet_name = clean_sheet_name(input("Name des Quell-Reiters (Alte Fassung): ").strip())

print("\nWie möchten Sie die NEUE Fassung einlesen?")
print("[1] Von BaFin-Webseite (URL)")
print("[2] Aus PDF-Dokument (Upload)")
source_choice = input("Bitte wählen Sie (1 oder 2): ").strip()

new_data_list = []
new_sheet_name = clean_sheet_name(input("\nName des Ziel-Reiters für die Neufassung: ").strip())

if source_choice == '1':
    url = input("URL der Neufassung: ").strip()
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    content_area = soup.find("div", class_="wrapBodytext") or soup

    current_abschnitt = "Unbekannt"
    for element in content_area.find_all(["h2", "h3", "h4", "ol"]):
        if element.name in ["h2", "h3", "h4"]:
            current_abschnitt = element.get_text(separator=" ", strip=True)
        elif element.name == "ol":
            if element.find_parent("ol"): continue
            tz_counter = 1
            for li in element.find_all("li", recursive=False):
                for ul in li.find_all(["ul", "ol"]):
                    for inner_li in ul.find_all("li"):
                        inner_li.insert_before(" • ")

                text = li.get_text(separator=" ", strip=True)
                if text: new_data_list.append({"Abschnitt": current_abschnitt, "TZ": str(tz_counter), "Inhalt MaRisk": clean_extracted_text(text), "Erläuterung": ""})
                tz_counter += 1
elif source_choice == '2':
    print("\nBitte laden Sie das PDF-Dokument der Neufassung hoch:")
    uploaded_pdf = files.upload()
    if uploaded_pdf:
        pdf_filename = list(uploaded_pdf.keys())[0]
        print("Analysiere PDF... Bitte warten.")
        new_data_list = extract_from_pdf(pdf_filename)
else:
    print("Ungültige Auswahl. Skript wird beendet.")
    exit()

# --- 2. VERGLEICH & VERARBEITUNG ---
cockpit_data, report_items = [], []

old_dict = {}
if master_filename and old_sheet_name:
    df_old = pd.read_excel(master_filename, sheet_name=old_sheet_name)
    df_old['TZ'] = df_old['TZ'].astype(str)
    old_dict = {generate_match_key(row['Abschnitt'], row['TZ']): clean_extracted_text(str(row['Inhalt MaRisk']).strip()) for _, row in df_old.iterrows()}

for item in new_data_list:
    abschnitt, tz, new_text, erlaeuterung = item['Abschnitt'], item['TZ'], item['Inhalt MaRisk'], item.get('Erläuterung', '')
    match_key = generate_match_key(abschnitt, tz)

    old_text = old_dict.get(match_key, "[Nicht vorhanden]")
    is_new = old_text == "[Nicht vorhanden]"
    has_changed = (not is_new) and (normalize_for_comparison(old_text) != normalize_for_comparison(new_text))

    status = "UNVERÄNDERT"
    if is_new: status = "NEU"
    elif has_changed: status = "GEÄNDERT"

    cockpit_data.append({"Abschnitt": abschnitt, "TZ": tz, "Status": status, "Inhalt NEU": new_text, "Erläuterung NEU": erlaeuterung, "Inhalt ALT": old_text, "Analyse": ""})
    report_items.append({"abs": abschnitt, "tz": tz, "old": old_text, "new": new_text, "erlaeuterung": erlaeuterung, "changed": has_changed, "is_new": is_new})

# --- 3. EXPORT EXCEL ---
output_file = master_filename or "MaRisk_Master.xlsx"
mode = 'a' if master_filename else 'w'
with pd.ExcelWriter(output_file, engine='openpyxl', mode=mode, if_sheet_exists='replace' if mode=='a' else None) as writer:
    pd.DataFrame(new_data_list).to_excel(writer, sheet_name=new_sheet_name, index=False)
    if cockpit_data: pd.DataFrame(cockpit_data).to_excel(writer, sheet_name="Differenzprotokoll", index=False)

# --- 4. HTML DASHBOARD GENERIERUNG ---
if report_items:
    toc_data = {}
    for item in report_items:
        if item['abs'] not in toc_data: toc_data[item['abs']] = []
        toc_data[item['abs']].append(item)

    toc_html_parts = []
    for abs_name, items in toc_data.items():
        changed_count = sum(1 for x in items if x['changed'] or x['is_new'])
        has_change_class = "has-change" if changed_count > 0 else ""
        sec_id = re.sub(r'\W+', '_', abs_name)

        toc_html_parts.append(f'<details class="toc-details" id="toc_det_{sec_id}"><summary class="toc-summary {has_change_class}">{abs_name} ({changed_count} Änderungen)<span class="assigned-depts-sec" id="toc_sec_{sec_id}"></span></summary><div class="toc-tz-list">')
        for it in items:
            uid = f"n_{it['abs']}_{it['tz']}".replace(" ","_")
            tz_class = "has-change" if it['changed'] or it['is_new'] else ""
            is_new_label = " (NEU)" if it['is_new'] else ""
            toc_html_parts.append(f'<a class="toc-tz-link {tz_class}" id="link_{uid}" href="#card_{uid}"><span style="display:flex; align-items:center;"><span class="prio-dot" id="dot_{uid}"></span>TZ {it["tz"]}{is_new_label}</span> <span class="assigned-depts" id="toc_dept_{uid}"></span></a>')
        toc_html_parts.append('</div></details>')

    toc_html_content = "".join(toc_html_parts)

    html_report = f"""
    <!DOCTYPE html>
    <html lang="de">
    <head>
        <meta charset="utf-8">
        <title>MaRisk-Novelle: Gap-Analyse</title>
        <style>
            /* Global UI/UX */
            body {{ font-family: 'Inter', 'Roboto', 'Segoe UI', sans-serif; margin: 0; padding: 40px; background: #f4f6f9; color: #2b2b2b; font-size: 14px; line-height: 1.6; }}
            .wrapper {{ max-width: 1200px; margin: auto; }}

            header {{ border-bottom: 2px solid #be0f34; margin-bottom: 20px; padding-bottom: 15px; }}
            .header-top {{ display: flex; justify-content: space-between; align-items: flex-end; margin-bottom: 15px; flex-wrap: wrap; gap: 15px; }}
            h1 {{ color: #be0f34; font-size: 22px; font-weight: 700; margin: 0; text-transform: uppercase; letter-spacing: 0.5px; }}

            /* Progress Bar */
            .progress-container {{ background: #e2e8f0; border-radius: 4px; height: 8px; overflow: hidden; width: 100%; }}
            #progress-bar {{ background: #be0f34; width: 0%; height: 100%; transition: width 0.4s ease-out; }}
            #progress-text {{ font-size: 11px; font-weight: 600; color: #4a5568; margin-top: 5px; text-align: right; }}

            /* Disclaimer & Cards */
            .disclaimer-box {{ background-color: #fffbeb; color: #854d0e; border-left: 4px solid #eab308; border-radius: 4px; padding: 15px 20px; margin-bottom: 25px; font-size: 13px; box-shadow: 0 2px 4px rgba(0,0,0,0.02); display: flex; align-items: flex-start; }}
            .instruction-box {{ background: #fff; border: none; border-radius: 8px; padding: 20px; margin-bottom: 25px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); transition: transform 0.2s ease-in-out; }}
            .instruction-box:hover {{ transform: translateY(-2px); }}

            .filter-bar {{ display: flex; gap: 15px; align-items: center; margin-bottom: 25px; padding: 15px 20px; background: #fff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); flex-wrap: wrap; }}
            .filter-title {{ font-weight: 700; color: #718096; font-size: 11px; letter-spacing: 0.5px; text-transform: uppercase; }}

            /* Buttons */
            .filter-btn {{ padding: 6px 16px; border: 1px solid transparent; background: #edf2f7; color: #4a5568; border-radius: 20px; cursor: pointer; font-size: 12px; font-weight: 600; transition: all 0.2s ease-in-out; }}
            .filter-btn:hover {{ background: #e2e8f0; transform: translateY(-1px); }}
            .filter-btn.active {{ background: #be0f34; color: #fff; box-shadow: 0 4px 6px rgba(190, 15, 52, 0.2); }}

            .btn-action {{ background: #fff; color: #4a5568; border: 1px solid #cbd5e1; padding: 8px 14px; border-radius: 6px; cursor: pointer; display: flex; align-items: center; gap: 6px; font-size: 13px; font-weight: 600; transition: all 0.2s ease; }}
            .btn-action:hover {{ border-color: #be0f34; color: #be0f34; background: #fff5f5; transform: translateY(-1px); }}
            .btn-action svg {{ width: 16px; height: 16px; }}

            .btn-exp {{ background: #be0f34; color: #fff; padding: 10px 18px; border: none; font-weight: 600; cursor: pointer; font-size: 13px; border-radius: 6px; transition: all 0.2s ease; box-shadow: 0 2px 4px rgba(190, 15, 52, 0.2); display: flex; align-items: center; gap: 6px; }}
            .btn-exp:hover {{ background: #9b0c2a; transform: translateY(-2px); box-shadow: 0 4px 8px rgba(190, 15, 52, 0.3); }}
            .btn-exp svg {{ width: 16px; height: 16px; }}

            .header-buttons {{ display: flex; gap: 10px; flex-wrap: wrap; }}

            /* TOC */
            details.toc-card {{ background: #fff; border: none; border-radius: 8px; padding: 20px; margin-bottom: 30px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); }}
            details.toc-card > summary {{ font-size: 12px; color: #718096; font-weight: 700; cursor: pointer; outline: none; margin-bottom: 10px; text-transform: uppercase; letter-spacing: 0.5px; }}
            .toc-details {{ margin-bottom: 6px; border: 1px solid #edf2f7; background: #f8fafc; border-radius: 6px; overflow: hidden; }}
            .toc-summary {{ padding: 10px 15px; cursor: pointer; color: #2d3748; font-weight: 600; font-size: 13px; list-style: none; display: flex; align-items: center; transition: background 0.2s; }}
            .toc-summary:hover {{ background: #edf2f7; }}
            .toc-summary::-webkit-details-marker {{ display: none; }}
            .toc-summary::before {{ content: "▶"; font-size: 10px; color: #a0aec0; display: inline-block; transition: transform 0.2s; margin-right: 8px; }}
            .toc-details[open] .toc-summary::before {{ transform: rotate(90deg); }}
            .toc-summary.has-change {{ color: #be0f34; }}
            .toc-tz-list {{ padding: 5px 0; display: flex; flex-direction: column; background: #fff; border-top: 1px solid #edf2f7; }}
            .toc-tz-link {{ color: #4a5568; text-decoration: none; font-size: 12px; padding: 8px 15px; border-bottom: 1px solid #f7fafc; display: flex; justify-content: space-between; align-items: center; transition: all 0.2s; }}
            .toc-tz-link:hover {{ background: #fdf2f4; color: #be0f34; padding-left: 20px; }}
            .toc-tz-link.has-change {{ font-weight: 600; color: #be0f34; }}

            /* Item Cards */
            .item-card {{ background: #fff; border: none; border-radius: 8px; margin-bottom: 25px; padding: 25px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); transition: all 0.3s ease; }}
            .item-card:hover {{ box-shadow: 0 6px 16px rgba(0,0,0,0.08); }}
            .is-unchanged {{ opacity: 0.7; }}
            .is-changed {{ border-left: 4px solid #be0f34; }}
            .is-new {{ border-left: 4px solid #10b981; background: #f0fdf4; }}
            .card-done {{ opacity: 0.6; background: #f8fafc; border-left-color: #cbd5e1; }}

            del {{ background: #fee2e2; color: #b91c1c; text-decoration: line-through; padding: 0 2px; border-radius: 2px; }}
            ins {{ background: #dcfce3; color: #047857; font-weight: 600; padding: 0 2px; border-radius: 2px; text-decoration: none; }}

            /* Details & Grids */
            details.volltext {{ margin-top: 15px; border: 1px solid #e2e8f0; border-radius: 6px; background: #f8fafc; overflow: hidden; }}
            details.volltext summary {{ padding: 12px 15px; cursor: pointer; color: #4a5568; font-size: 12px; font-weight: 600; outline: none; }}
            details.volltext summary:hover {{ background: #edf2f7; }}
            .details-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; padding: 15px; background: #fff; border-top: 1px solid #e2e8f0; }}
            .details-col {{ font-size: 13px; color: #4a5568; }}

            /* Controls & Inputs */
            .controls-panel {{ margin-top: 20px; border-top: 1px solid #e2e8f0; padding-top: 20px; }}
            .control-row {{ display: flex; gap: 20px; flex-wrap: wrap; align-items: flex-start; margin-bottom: 15px; }}
            .ctrl-group {{ display: flex; flex-direction: column; gap: 6px; flex-grow: 1; }}
            .ctrl-label {{ font-size: 11px; font-weight: 700; color: #718096; text-transform: uppercase; letter-spacing: 0.5px; }}

            select, .team-input, .search-input {{ padding: 8px 12px; border: 1px solid #cbd5e1; border-radius: 4px; font-size: 13px; font-family: inherit; color: #2d3748; background: #fff; outline: none; transition: border-color 0.2s, box-shadow 0.2s; box-sizing: border-box; }}
            .team-input {{ width: 100%; max-width: 300px; }}
            .search-input {{ width: 200px; }}
            select:focus, .team-input:focus, .search-input:focus {{ border-color: #be0f34; box-shadow: 0 0 0 1px #be0f34; }}

            /* --- WYSIWYG TEXT EDITOR --- */
            .editor-container {{ border: 1px solid #e0e0e0; border-radius: 4px; overflow: hidden; background: #fff; transition: border-color 0.2s, box-shadow 0.2s; }}
            .editor-container:focus-within {{ border-color: #be0f34; box-shadow: 0 0 0 1px #be0f34; }}

            .editor-toolbar {{ display: flex; gap: 4px; padding: 8px; background: #ffffff; border-bottom: 1px solid #e0e0e0; flex-wrap: wrap; align-items: center; }}
            .toolbar-btn {{ background: transparent; border: none; padding: 6px; border-radius: 4px; cursor: pointer; color: #111111; font-size: 13px; font-weight: 600; transition: background 0.2s, color 0.2s; display: flex; align-items: center; justify-content: center; }}
            .toolbar-btn:hover {{ background: #f4f6f9; color: #be0f34; }}
            .toolbar-divider {{ width: 1px; height: 18px; background: #e0e0e0; margin: 0 6px; }}

            .toolbar-select {{ padding: 4px 8px; border: 1px solid #e0e0e0; border-radius: 4px; font-size: 12px; background: #fff; color: #111111; cursor: pointer; outline: none; transition: border-color 0.2s; }}
            .toolbar-select:focus {{ border-color: #be0f34; }}

            .analysis-input {{ min-height: 100px; padding: 15px; font-size: 13px; line-height: 1.6; color: #111111; outline: none; overflow-y: auto; background: #fff; }}
            .analysis-input p {{ margin: 0 0 8px 0; }}
            .analysis-input ul, .analysis-input ol {{ margin: 0 0 10px 0; padding-left: 20px; }}
            .analysis-input a {{ color: #00376C; text-decoration: underline; }}
            .analysis-input h1, .analysis-input h2, .analysis-input h3 {{ margin: 15px 0 8px 0; color: #111111; font-weight: 700; }}

            /* Status Dots */
            .prio-dot {{ display: inline-block; width: 8px; height: 8px; border-radius: 50%; margin-right: 6px; display: none; }}
            .dot-Hoch {{ background: #be0f34; display: inline-block; }}
            .dot-Mittel {{ background: #f59e0b; display: inline-block; }}
            .dot-Gering {{ background: #10b981; display: inline-block; }}
            .assigned-depts, .assigned-depts-sec {{ color: #4a5568; font-weight: 600; font-size: 11px; margin-left: 8px; }}

            /* SVG Helpers */
            .icon-svg {{ width: 16px; height: 16px; stroke: currentColor; stroke-width: 2; fill: none; stroke-linecap: round; stroke-linejoin: round; }}
            .icon-svg-text {{ width: 18px; height: 18px; stroke: currentColor; stroke-width: 2.5; fill: none; stroke-linecap: round; stroke-linejoin: round; }}
        </style>
    </head>
    <body id="top"><div class="wrapper">
        <header>
            <div class="header-top">
                <h1>MaRisk-Novelle: Gap-Analyse V12.14</h1>
                <div class="header-buttons">
                    <button class="btn-action" onclick="resetText()" title="Analysetexte löschen">
                        <svg viewBox="0 0 24 24" class="icon-svg"><polyline points="3 6 5 6 21 6"></polyline><path d="M19 6v14a2 2 0 0 1-2 2H7a2 2 0 0 1-2-2V6m3 0V4a2 2 0 0 1 2-2h4a2 2 0 0 1 2 2v2"></path></svg>
                        Texte löschen
                    </button>
                    <button class="btn-action" onclick="resetAll()" title="Zuweisungen zurücksetzen">
                        <svg viewBox="0 0 24 24" class="icon-svg"><polyline points="23 4 23 10 17 10"></polyline><polyline points="1 20 1 14 7 14"></polyline><path d="M3.51 9a9 9 0 0 1 14.85-3.36L23 10M1 14l4.64 4.36A9 9 0 0 0 20.49 15"></path></svg>
                        Zuweisungen zurücksetzen
                    </button>
                    <input type="file" id="json-upload" accept=".json" style="display:none" onchange="importJSON(event)">
                    <button class="btn-action" onclick="document.getElementById('json-upload').click()">
                        <svg viewBox="0 0 24 24" class="icon-svg"><path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path><polyline points="17 8 12 3 7 8"></polyline><line x1="12" y1="3" x2="12" y2="15"></line></svg>
                        Stand laden
                    </button>
                    <button class="btn-action" onclick="exportJSON()">
                        <svg viewBox="0 0 24 24" class="icon-svg"><path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path><polyline points="7 10 12 15 17 10"></polyline><line x1="12" y1="15" x2="12" y2="3"></line></svg>
                        Stand speichern
                    </button>
                    <button class="btn-exp" onclick="exportCSV()">
                        <svg viewBox="0 0 24 24" class="icon-svg"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"></path><polyline points="14 2 14 8 20 8"></polyline><line x1="16" y1="13" x2="8" y2="13"></line><line x1="16" y1="17" x2="8" y2="17"></line><polyline points="10 9 9 9 8 9"></polyline></svg>
                        CSV Export
                    </button>
                </div>
            </div>
            <div class="progress-container">
                <div id="progress-bar"></div>
            </div>
            <div id="progress-text">0 von 0 relevanten Änderungen bearbeitet (0%)</div>
        </header>

        <div class="disclaimer-box">
            <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" style="min-width:18px; height:18px; margin-right:10px; margin-top:2px;">
                <path d="M10.29 3.86L1.82 18a2 2 0 0 0 1.71 3h16.94a2 2 0 0 0 1.71-3L13.71 3.86a2 2 0 0 0-3.42 0z"></path><line x1="12" y1="9" x2="12" y2="13"></line><line x1="12" y1="17" x2="12.01" y2="17"></line>
            </svg>
            <div>
                <strong>Wichtiger Hinweis:</strong> Die hier dargestellten Texte wurden maschinell aus PDF-Dokumenten extrahiert. Trotz intelligenter Filteralgorithmen kann es im Zuge der Umwandlung zu Abweichungen oder Formatierungsfehlern kommen. Es wird keine Gewähr für die Vollständigkeit oder rechtliche Richtigkeit der extrahierten Daten übernommen. Bitte gleiche kritische Stellen stets mit den Originaldokumenten ab.
            </div>
        </div>

        <div class="instruction-box">
            <div style="font-weight:700; color:#be0f34; margin-bottom:12px; font-size:15px; border-bottom:1px solid #edf2f7; padding-bottom:8px;">
                Gegenüberstellung: {old_sheet_name if old_sheet_name else 'Keine Alt-Fassung'} vs. {new_sheet_name}
            </div>
            <ul style="margin:5px 0 0; padding-left:20px; color:#4a5568; font-size: 13px;">
                <li><b>Filtern & Suchen:</b> Nutze die Leiste unten, um gezielt nach Stichworten oder Status zu filtern. Blende unveränderte Passagen einfach aus.</li>
                <li><b>Navigation:</b> Das interaktive Inhaltsverzeichnis bringt dich mit einem Klick direkt zur gewünschten Textziffer.</li>
                <li><b>Datensicherung:</b> Nutze die Funktionen im Header, um deine Arbeit als JSON zu speichern, Stände zu laden oder das finale Ergebnis als CSV zu exportieren.</li>
            </ul>
        </div>

        <div class="filter-bar">
            <span class="filter-title">SUCHE:</span>
            <input type="text" id="search-filter" class="search-input" placeholder="Stichwort..." oninput="applyFilters()">

            <span class="filter-title" style="margin-left: 5px;">STATUS:</span>
            <select id="status-filter" onchange="applyFilters()">
                <option value="ALL">Alle</option>
                <option value="Offen">Offen</option>
                <option value="In Prüfung">In Prüfung</option>
                <option value="In Umsetzung">In Umsetzung</option>
                <option value="Abgeschlossen">Abgeschlossen</option>
            </select>

            <label style="margin-left: 10px; font-size: 12px; font-weight: 600; color: #718096; display: flex; align-items: center; gap: 6px; cursor: pointer;">
                <input type="checkbox" id="changes-only-toggle" onchange="applyFilters()">
                Nur Änderungen zeigen
            </label>

            <div style="width: 100%; height: 1px; background: #edf2f7; margin: 5px 0;"></div>

            <span class="filter-title">TEAMS:</span>
            <button id="btn-filter-all" class="filter-btn active" onclick="toggleFilter('ALL', this)">Alle anzeigen</button>
            <div id="dynamic-filters" style="display:inline-flex; gap:10px; flex-wrap:wrap;"></div>
        </div>

        <details class="toc-card" open>
            <summary>INHALTSVERZEICHNIS (Ganzes Verzeichnis ein-/ausklappen)</summary>
            <div style="margin-top: 15px;">
                {toc_html_content}
            </div>
        </details>
    """

    for item in report_items:
        uid = f"n_{item['abs']}_{item['tz']}".replace(" ","_")
        sec_id = re.sub(r'\W+', '_', item['abs'])
        card_class = "is-new" if item['is_new'] else ("is-changed" if item['changed'] else "is-unchanged")
        diff_html = format_text_with_lists(item['new'] if (item['is_new'] or not item['changed']) else generate_inline_diff(item['old'], item['new']))

        safe_old = html.escape(str(item['old']))
        safe_new = html.escape(str(item['new']))

        erlaeuterung_html = ""
        if item['erlaeuterung']:
            erl_formatted = format_erlaeuterung_with_html(str(item['erlaeuterung']))
            safe_erl_raw = html.escape(str(item['erlaeuterung']))
            erlaeuterung_html = f"""
            <details style="margin-top: 15px; border-left: 4px solid #a0aec0; background: #f8fafc; border-radius: 0 4px 4px 0; overflow: hidden;" open>
                <summary style="padding: 12px 15px; cursor: pointer; color: #4a5568; font-size: 12px; font-weight: 700; text-transform: uppercase; list-style: none; display: flex; align-items: center; background: #edf2f7; transition: background 0.2s;">
                    <svg viewBox="0 0 24 24" class="icon-svg" style="margin-right:8px;"><circle cx="12" cy="12" r="10"></circle><line x1="12" y1="16" x2="12" y2="12"></line><line x1="12" y1="8" x2="12.01" y2="8"></line></svg>
                    BaFin Erläuterungen
                </summary>
                <div id="erl_{uid}" style="padding: 15px 20px; font-size: 13px; line-height: 1.6; color: #2d3748; background: #f8fafc;">
                    {erl_formatted}
                </div>
            </details>
            """
        else:
            safe_erl_raw = ""

        details_html = ""
        if item['changed']:
            details_html = f"""
            <details class="volltext">
                <summary>VOLLTEXT-GEGENÜBERSTELLUNG EINBLENDEN</summary>
                <div class="details-grid">
                    <div class="details-col"><strong>Fassung Alt:</strong><br><br>{format_text_with_lists(item['old'])}</div>
                    <div class="details-col"><strong>Fassung Neu:</strong><br><br>{format_text_with_lists(item['new'])}</div>
                </div>
            </details>
            """

        html_report += f"""
        <div id="card_{uid}" class="item-card {card_class}" data-abs="{item['abs']}" data-tz="{item['tz']}" data-old="{safe_old}" data-new="{safe_new}" data-erl="{safe_erl_raw}">
            <a href="#top" style="float:right; font-size:10px; color:#a0aec0; text-decoration:none; font-weight: 700; display:flex; align-items:center;">TOP <svg viewBox="0 0 24 24" style="width:12px; height:12px; margin-left:2px; stroke:currentColor; stroke-width:2; fill:none;"><polyline points="18 15 12 9 6 15"></polyline></svg></a>
            <div class="item-title" style="font-weight:700; color:#be0f34; font-size: 15px; margin-bottom:10px;">{item['abs']} – TZ {item['tz']} {"<span style='color:#10b981; margin-left:5px;'>(NEU)</span>" if item['is_new'] else ""}</div>
            <div class="diff-html-content" style="color: #2d3748;">{diff_html}</div>
            {erlaeuterung_html}
            {details_html}

            <div class="controls-panel">
                <div class="control-row">
                    <div class="ctrl-group" style="flex: 2;">
                        <span class="ctrl-label">Teams / Zuständigkeit</span>
                        <input type="text" id="team_{uid}" data-sec="{sec_id}" class="team-input" oninput="saveState()" placeholder="z.B. IT, Compliance...">
                    </div>
                    <div class="ctrl-group">
                        <span class="ctrl-label">Status</span>
                        <select id="stat_{uid}" class="stat-sel" onchange="saveState()">
                            <option value="Offen">Offen</option>
                            <option value="In Prüfung">In Prüfung</option>
                            <option value="In Umsetzung">In Umsetzung</option>
                            <option value="Abgeschlossen">Abgeschlossen</option>
                        </select>
                    </div>
                    <div class="ctrl-group">
                        <span class="ctrl-label">Priorität</span>
                        <select id="prio_{uid}" class="prio-sel" onchange="saveState()">
                            <option value="Keine">Keine Zuordnung</option>
                            <option value="Hoch">Hoch</option>
                            <option value="Mittel">Mittel</option>
                            <option value="Gering">Gering</option>
                        </select>
                    </div>
                </div>

                <div class="ctrl-group" style="margin-top: 5px;">
                    <span class="ctrl-label">Fachliche Analyse & Maßnahmen</span>

                    <div class="editor-container">
                        <div class="editor-toolbar">
                            <select class="toolbar-select" onchange="execCmd('formatBlock', this.value, '{uid}'); this.selectedIndex=0;" title="Formatierung">
                                <option value="">Format...</option>
                                <option value="P">Text</option>
                                <option value="H2">Überschrift</option>
                                <option value="H3">Unterüberschrift</option>
                            </select>
                            <div class="toolbar-divider"></div>
                            <button type="button" class="toolbar-btn" onclick="execCmd('bold', null, '{uid}')" title="Fett">
                                <svg class="icon-svg-text" viewBox="0 0 24 24"><path d="M6 4h8a4 4 0 0 1 4 4 4 4 0 0 1-4 4H6z"></path><path d="M6 12h9a4 4 0 0 1 4 4 4 4 0 0 1-4 4H6z"></path></svg>
                            </button>
                            <button type="button" class="toolbar-btn" onclick="execCmd('italic', null, '{uid}')" title="Kursiv">
                                <svg class="icon-svg-text" viewBox="0 0 24 24"><line x1="19" y1="4" x2="10" y2="4"></line><line x1="14" y1="20" x2="5" y2="20"></line><line x1="15" y1="4" x2="9" y2="20"></line></svg>
                            </button>
                            <button type="button" class="toolbar-btn" onclick="execCmd('underline', null, '{uid}')" title="Unterstrichen">
                                <svg class="icon-svg-text" viewBox="0 0 24 24"><path d="M6 3v7a6 6 0 0 0 6 6 6 6 0 0 0 6-6V3"></path><line x1="4" y1="21" x2="20" y2="21"></line></svg>
                            </button>
                            <div class="toolbar-divider"></div>
                            <button type="button" class="toolbar-btn" onclick="execCmd('insertUnorderedList', null, '{uid}')" title="Aufzählung">
                                <svg class="icon-svg" viewBox="0 0 24 24"><line x1="8" y1="6" x2="21" y2="6"></line><line x1="8" y1="12" x2="21" y2="12"></line><line x1="8" y1="18" x2="21" y2="18"></line><line x1="3" y1="6" x2="3.01" y2="6"></line><line x1="3" y1="12" x2="3.01" y2="12"></line><line x1="3" y1="18" x2="3.01" y2="18"></line></svg>
                            </button>
                            <button type="button" class="toolbar-btn" onclick="execCmd('insertOrderedList', null, '{uid}')" title="Nummerierung">
                                <svg class="icon-svg" viewBox="0 0 24 24"><line x1="10" y1="6" x2="21" y2="6"></line><line x1="10" y1="12" x2="21" y2="12"></line><line x1="10" y1="18" x2="21" y2="18"></line><path d="M4 6h1v4"></path><path d="M4 10h2"></path><path d="M6 18H4c0-1 2-2 2-3s-1-1.5-2-1"></path></svg>
                            </button>
                            <div class="toolbar-divider"></div>
                            <button type="button" class="toolbar-btn" onclick="addLink('{uid}')" title="Link einfügen">
                                <svg class="icon-svg" viewBox="0 0 24 24"><path d="M10 13a5 5 0 0 0 7.54.54l3-3a5 5 0 0 0-7.07-7.07l-1.72 1.71"></path><path d="M14 11a5 5 0 0 0-7.54-.54l-3 3a5 5 0 0 0 7.07 7.07l1.71-1.71"></path></svg>
                            </button>
                            <div class="toolbar-divider"></div>
                            <button type="button" class="toolbar-btn" onclick="execCmd('foreColor', '#111111', '{uid}')" title="Textfarbe Schwarz">
                                <svg class="icon-svg" viewBox="0 0 24 24"><circle cx="12" cy="12" r="7" fill="#111111" stroke="none"></circle></svg>
                            </button>
                            <button type="button" class="toolbar-btn" onclick="execCmd('foreColor', '#00376C', '{uid}')" title="Textfarbe Corporate Blue">
                                 <svg class="icon-svg" viewBox="0 0 24 24"><circle cx="12" cy="12" r="7" fill="#00376C" stroke="none"></circle></svg>
                            </button>
                            <button type="button" class="toolbar-btn" onclick="execCmd('foreColor', '#be0f34', '{uid}')" title="Textfarbe Brand Red">
                                 <svg class="icon-svg" viewBox="0 0 24 24"><circle cx="12" cy="12" r="7" fill="#be0f34" stroke="none"></circle></svg>
                            </button>
                        </div>
                        <div id="text_{uid}" class="analysis-input" contenteditable="true" onblur="saveState()" oninput="updateProgress()"></div>
                    </div>

                </div>
            </div>
        </div>
        """

    html_report += """</div><script>
        window.onload = () => {
            document.querySelectorAll('.stat-sel').forEach(sel => sel.value = localStorage.getItem(sel.id) || 'Offen');
            document.querySelectorAll('.prio-sel').forEach(sel => sel.value = localStorage.getItem(sel.id) || 'Keine');
            document.querySelectorAll('.team-input').forEach(inp => inp.value = localStorage.getItem(inp.id) || '');

            document.querySelectorAll('.analysis-input').forEach(el => {
                let saved = localStorage.getItem(el.id);
                if(saved !== null) el.innerHTML = saved;
            });

            updateUI();
        };

        function exportJSON() {
            let data = {};
            document.querySelectorAll('.item-card').forEach(card => {
                let uid = card.id.replace('card_', '');
                data[uid] = {
                    team: document.getElementById('team_' + uid).value,
                    status: document.getElementById('stat_' + uid).value,
                    prio: document.getElementById('prio_' + uid).value,
                    analysis: document.getElementById('text_' + uid).innerHTML
                };
            });
            let blob = new Blob([JSON.stringify(data, null, 2)], {type: "application/json"});
            let link = document.createElement("a");
            link.href = URL.createObjectURL(blob);
            link.download = "MaRisk_Projektstand.json";
            link.click();
        }

        function importJSON(event) {
            let file = event.target.files[0];
            if (!file) return;
            let reader = new FileReader();
            reader.onload = function(e) {
                try {
                    let data = JSON.parse(e.target.result);
                    for (let uid in data) {
                        if (document.getElementById('team_' + uid)) document.getElementById('team_' + uid).value = data[uid].team || "";
                        if (document.getElementById('stat_' + uid)) document.getElementById('stat_' + uid).value = data[uid].status || "Offen";
                        if (document.getElementById('prio_' + uid)) document.getElementById('prio_' + uid).value = data[uid].prio || "Keine";
                        if (document.getElementById('text_' + uid)) document.getElementById('text_' + uid).innerHTML = data[uid].analysis || "";
                    }
                    saveState();
                    alert("Projektstand erfolgreich geladen!");
                } catch(err) {
                    alert("Fehler beim Laden der Datei. Ist es ein valides JSON?");
                }
            };
            reader.readAsText(file);
            event.target.value = '';
        }

        function execCmd(command, value, uid) {
            const editor = document.getElementById('text_' + uid);
            editor.focus();
            document.execCommand(command, false, value);
            saveState();
        }

        function addLink(uid) {
            const url = prompt('Bitte Link-URL eingeben (z.B. https://...):', 'https://');
            if (url) execCmd('createLink', url, uid);
        }

        function getTeamsFromString(str) {
            if(!str) return [];
            return str.split(',').map(t => t.trim()).filter(t => t.length > 0);
        }

        function saveState() {
            document.querySelectorAll('.stat-sel, .prio-sel, .team-input').forEach(el => localStorage.setItem(el.id, el.value));
            document.querySelectorAll('.analysis-input').forEach(el => localStorage.setItem(el.id, el.innerHTML));
            updateUI();
        }

        function updateProgress() {
            let total = 0;
            let completed = 0;

            document.querySelectorAll('.item-card').forEach(card => {
                if (card.classList.contains('is-changed') || card.classList.contains('is-new')) {
                    total++;
                    let uid = card.id.replace('card_', '');
                    let status = document.getElementById('stat_' + uid).value;
                    let textContent = document.getElementById('text_' + uid).innerText.trim();

                    if(status !== 'Offen' || textContent.length > 0) {
                        completed++;
                    }
                }
            });

            let percent = total === 0 ? 0 : Math.round((completed / total) * 100);
            document.getElementById('progress-bar').style.width = percent + '%';
            document.getElementById('progress-text').innerText = completed + ' von ' + total + ' relevanten Änderungen bearbeitet (' + percent + '%)';
        }

        function toggleFilter(team, btnElement) {
            if (team === 'ALL') {
                document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
                btnElement.classList.add('active');
            } else {
                document.getElementById('btn-filter-all').classList.remove('active');
                btnElement.classList.toggle('active');

                if(document.querySelectorAll('#dynamic-filters .filter-btn.active').length === 0) {
                    document.getElementById('btn-filter-all').classList.add('active');
                }
            }
            applyFilters();
        }

        function resetAll() {
            if(!confirm("Möchtest du wirklich alle Zuweisungen, Status und Prioritäten zurücksetzen? (Deine Analysetexte bleiben erhalten)")) return;
            document.querySelectorAll('.stat-sel').forEach(sel => { sel.value = 'Offen'; localStorage.removeItem(sel.id); });
            document.querySelectorAll('.prio-sel').forEach(sel => { sel.value = 'Keine'; localStorage.removeItem(sel.id); });
            document.querySelectorAll('.team-input').forEach(inp => { inp.value = ''; localStorage.removeItem(inp.id); });
            document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
            document.getElementById('btn-filter-all').classList.add('active');
            updateUI();
        }

        function resetText() {
            if(!confirm("Möchtest du wirklich alle fachlichen Analysetexte unwiderruflich löschen?")) return;
            document.querySelectorAll('.analysis-input').forEach(el => { el.innerHTML = ''; localStorage.removeItem(el.id); });
            updateProgress();
        }

        function updateUI() {
            const tzRegistry = {};
            const secRegistry = {};
            const allUniqueTeams = new Set();

            document.querySelectorAll('.team-input').forEach(inp => {
                let uid = inp.id.replace('team_', '');
                let secId = inp.dataset.sec;
                let teams = getTeamsFromString(inp.value);

                if(!tzRegistry[uid]) tzRegistry[uid] = new Set();
                if(!secRegistry[secId]) secRegistry[secId] = new Set();

                teams.forEach(t => {
                    tzRegistry[uid].add(t);
                    secRegistry[secId].add(t);
                    allUniqueTeams.add(t);
                });
            });

            let dynContainer = document.getElementById('dynamic-filters');
            let currentActive = Array.from(dynContainer.querySelectorAll('.active')).map(b => b.innerText);
            dynContainer.innerHTML = '';

            let sortedTeams = Array.from(allUniqueTeams).sort();
            sortedTeams.forEach(t => {
                let isActive = currentActive.includes(t) ? 'active' : '';
                dynContainer.innerHTML += `<button class="filter-btn ${isActive}" onclick="toggleFilter('${t}', this)">${t}</button>`;
            });

            if(dynContainer.querySelectorAll('.active').length === 0 && !document.getElementById('btn-filter-all').classList.contains('active')) {
                document.getElementById('btn-filter-all').classList.add('active');
            }

            document.querySelectorAll('.assigned-depts, .assigned-depts-sec').forEach(el => el.innerText = '');

            for(let id in tzRegistry) {
                const span = document.getElementById('toc_dept_' + id);
                if(span && tzRegistry[id].size > 0) span.innerText = '[' + Array.from(tzRegistry[id]).join(', ') + ']';
            }

            for(let id in secRegistry) {
                const span = document.getElementById('toc_sec_' + id);
                if(span && secRegistry[id].size > 0) span.innerText = ' [' + Array.from(secRegistry[id]).join(', ') + ']';
            }

            document.querySelectorAll('.prio-dot').forEach(el => { el.className = 'prio-dot'; el.style.display = 'none'; });
            document.querySelectorAll('.toc-tz-link').forEach(el => el.classList.remove('link-done'));
            document.querySelectorAll('.item-card').forEach(el => el.classList.remove('card-done'));

            document.querySelectorAll('.item-card').forEach(card => {
                let uid = card.id.replace('card_', '');
                let status = document.getElementById('stat_' + uid).value;
                let prio = document.getElementById('prio_' + uid).value;
                let link = document.getElementById('link_' + uid);
                let dot = document.getElementById('dot_' + uid);

                if(status === 'Abgeschlossen') {
                    card.classList.add('card-done');
                    if(link) link.classList.add('link-done');
                }

                if(prio !== 'Keine' && dot) {
                    dot.classList.add('dot-' + prio);
                    dot.style.display = 'inline-block';
                }
            });

            applyFilters();
            updateProgress();
        }

        function applyFilters() {
            let activeFilters = Array.from(document.querySelectorAll('#dynamic-filters .filter-btn.active')).map(b => b.innerText);
            let showAll = document.getElementById('btn-filter-all').classList.contains('active');

            let searchText = document.getElementById('search-filter').value.toLowerCase();
            let statusFilter = document.getElementById('status-filter').value;
            let changesOnly = document.getElementById('changes-only-toggle').checked;

            document.querySelectorAll('.item-card').forEach(card => {
                let uid = card.id.replace('card_', '');
                let link = document.getElementById('link_' + uid);

                let teamsInCard = getTeamsFromString(document.getElementById('team_' + uid).value);
                let status = document.getElementById('stat_' + uid).value;
                let isChanged = card.classList.contains('is-changed') || card.classList.contains('is-new');

                let teamMatch = showAll || teamsInCard.some(t => activeFilters.includes(t));
                let statusMatch = (statusFilter === 'ALL' || status === statusFilter);
                let changesMatch = (!changesOnly || isChanged);

                let searchMatch = true;
                if(searchText) {
                    searchMatch = card.innerText.toLowerCase().includes(searchText);
                }

                let isVisible = teamMatch && statusMatch && changesMatch && searchMatch;

                card.style.display = isVisible ? 'block' : 'none';
                if(link) link.style.display = isVisible ? 'flex' : 'none';
            });

            document.querySelectorAll('.toc-details').forEach(det => {
                let visibleLinks = Array.from(det.querySelectorAll('.toc-tz-link')).filter(l => l.style.display !== 'none');
                det.style.display = (visibleLinks.length > 0) ? 'block' : 'none';
            });
        }

        function exportCSV() {
            let csv = "Abschnitt;Textziffer;Status;Prioritaet;Teams;Inhalt ALT;Inhalt NEU;Aenderungs-Diff;Erlaeuterung NEU;Analyse\\r\\n";

            let parseDiffForCSV = (htmlStr) => {
                if (!htmlStr) return "";
                let text = htmlStr;
                // Rote und Grüne Highlights in robuste VBA-Tags übersetzen
                text = text.replace(/<del[^>]*>/gi, '[DEL]');
                text = text.replace(/<\\/del>/gi, '[/DEL]');
                text = text.replace(/<ins[^>]*>/gi, '[INS]');
                text = text.replace(/<\\/ins>/gi, '[/INS]');

                // HTML-Struktur in sichere Zeilenumbrüche für Excel übersetzen
                text = text.replace(/<\\/p>|<br>|<\\/div>|<\\/li>|<\\/h[1-6]>/gi, '\\n');
                text = text.replace(/<li>/gi, '• ');
                text = text.replace(/<[^>]+>/g, ''); // Alle anderen HTML-Tags löschen

                let txt = document.createElement("textarea");
                txt.innerHTML = text;
                text = txt.value;

                // Überflüssige Leerzeichen löschen, aber Zeilenumbrüche erhalten
                text = text.replace(/[^\\S\\r\\n]+/g, ' ');
                return text.replace(/"/g, '""').trim();
            };

            let parseStandardForCSV = (htmlStr) => {
                if (!htmlStr) return "";
                let text = htmlStr;
                text = text.replace(/<\\/p>|<br>|<\\/div>|<\\/li>|<\\/h[1-6]>/gi, '\\n');
                text = text.replace(/<li>/gi, '• ');
                text = text.replace(/<[^>]+>/g, '');

                let txt = document.createElement("textarea");
                txt.innerHTML = text;
                text = txt.value;

                text = text.replace(/[^\\S\\r\\n]+/g, ' ');
                return text.replace(/"/g, '""').trim();
            };

            document.querySelectorAll('.item-card').forEach(card => {
                let abs = card.dataset.abs || "";
                let tz = card.dataset.tz || "";
                let uid = card.id.replace('card_', '');

                let statusEl = document.getElementById('stat_' + uid);
                let prioEl = document.getElementById('prio_' + uid);
                let teamEl = document.getElementById('team_' + uid);

                let status = statusEl ? statusEl.value : "Offen";
                let prio = prioEl ? prioEl.value : "Keine";
                let teams = teamEl ? teamEl.value.trim() : "";

                let oldText = parseStandardForCSV(card.dataset.old);
                let newText = parseStandardForCSV(card.dataset.new);
                let erlText = parseStandardForCSV(card.dataset.erl);

                let diffEl = card.querySelector('.diff-html-content');
                let diffText = diffEl ? parseDiffForCSV(diffEl.innerHTML) : "";

                let anaEl = document.getElementById('text_' + uid);
                let text = anaEl ? parseStandardForCSV(anaEl.innerHTML) : "";

                // Nur relevante Zeilen exportieren
                if(status !== 'Offen' || prio !== 'Keine' || teams || text || card.classList.contains('is-changed') || card.classList.contains('is-new')) {
                    csv += `"${abs}";"${tz}";"${status}";"${prio}";"${teams}";"${oldText}";"${newText}";"${diffText}";"${erlText}";"${text}"\\r\\n`;
                }
            });

            let blob = new Blob(["\\uFEFF" + csv], { type: 'text/csv;charset=utf-8;' });
            let link = document.createElement('a');
            link.href = URL.createObjectURL(blob); link.download = 'MaRisk_Massnahmenplan.csv'; link.click();
        }
    </script></body></html>"""

    with open("MaRisk_Differenzprotokoll.html", "w", encoding="utf-8") as f: f.write(html_report)
    files.download("MaRisk_Differenzprotokoll.html")

files.download(output_file)

MaRisk-Novelle: Gap-Analyse V12.14 (Excel VBA Edition)
Bestands-Masterdatei (alte Fassung) hochladen? (j/n): j

Bitte laden Sie die Excel-Masterdatei hoch:


Saving MaRisk_Master.xlsx to MaRisk_Master (7).xlsx
Name des Quell-Reiters (Alte Fassung): OG

Wie möchten Sie die NEUE Fassung einlesen?
[1] Von BaFin-Webseite (URL)
[2] Aus PDF-Dokument (Upload)
Bitte wählen Sie (1 oder 2): 2

Name des Ziel-Reiters für die Neufassung: test

Bitte laden Sie das PDF-Dokument der Neufassung hoch:


Saving dl_kon_02_2026_rs_marisk-novelle_konsultationsfassung.pdf to dl_kon_02_2026_rs_marisk-novelle_konsultationsfassung (7).pdf
Analysiere PDF... Bitte warten.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>